# yeast6: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('44_yeast6_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,37.783,28.990,70.000,20.000,99.931,59.965,60.202,83.924,33.387
smote,82.706,47.210,35.556,71.429,96.860,84.144,55.238,91.805,64.958
bsmote,80.871,48.180,37.945,68.571,97.309,82.940,50.803,93.279,64.525
ros,84.918,45.813,33.114,75.714,96.308,86.011,49.139,91.646,65.366
adasyn,82.269,41.871,29.941,71.429,95.790,83.609,54.335,91.567,62.070
casb,84.534,49.580,37.264,74.286,96.998,85.642,51.547,91.549,67.057
casb_relaxed,81.820,48.152,37.151,70.000,97.136,83.568,55.504,91.820,64.986
casb_edited,86.685,48.793,35.768,78.571,96.515,87.543,56.002,91.967,67.739
casb_ros,85.255,50.379,37.875,75.714,97.032,86.373,50.862,92.037,67.817


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,72.459,58.549,66.190,54.286,99.309,76.798,52.540,88.314,65.504
smote,82.723,33.180,21.471,74.286,93.237,83.761,33.187,88.263,57.952
bsmote,82.730,45.874,33.971,71.429,96.584,84.006,36.547,87.983,64.302
ros,83.576,39.717,27.146,74.286,95.169,84.728,32.521,87.742,61.646
adasyn,82.230,30.143,19.004,74.286,92.167,83.227,30.909,89.329,56.187
casb,83.498,39.012,26.467,74.286,94.962,84.624,33.320,87.537,61.255
casb_relaxed,83.205,36.583,24.339,74.286,94.306,84.296,37.130,89.312,59.894
casb_edited,84.224,32.736,20.840,77.143,92.719,84.931,33.711,89.145,58.480
casb_ros,83.576,39.717,27.146,74.286,95.169,84.728,33.082,87.771,61.646


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,49.265,42.939,73.333,31.429,99.896,65.662,58.804,92.000,46.102
smote,77.063,56.462,54.480,62.857,98.723,80.790,58.297,93.536,66.762
bsmote,74.362,52.856,52.290,58.571,98.654,78.613,55.792,92.950,63.609
ros,75.610,57.418,56.457,61.429,98.930,80.179,59.909,91.794,66.514
adasyn,79.318,55.502,50.230,65.714,98.378,82.046,55.684,93.570,67.410
casb,73.305,54.771,54.730,57.143,98.930,78.036,59.069,93.001,64.038
casb_relaxed,73.232,52.673,50.070,57.143,98.723,77.933,56.631,92.469,62.952
casb_edited,78.852,49.142,39.956,65.714,97.654,81.684,58.367,94.084,63.997
casb_ros,74.069,56.518,57.169,58.571,98.999,78.785,59.458,92.245,65.294


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,52.850,41.323,72.667,31.429,99.724,65.576,50.959,94.432,47.086
smote,85.723,34.122,21.848,80.000,92.994,86.497,48.429,94.382,59.922
bsmote,81.015,38.092,26.097,71.429,95.307,83.368,43.711,94.161,59.553
ros,85.821,34.060,21.781,80.000,92.925,86.462,45.836,94.268,59.941
adasyn,86.981,32.039,19.985,82.857,91.717,87.287,45.810,94.175,59.510
casb,85.416,37.266,24.609,78.571,94.203,86.387,49.444,94.501,61.341
casb_relaxed,86.111,36.694,24.014,80.000,93.822,86.911,48.732,94.377,61.402
casb_edited,86.045,35.424,22.940,80.000,93.443,86.721,49.037,94.392,60.734
casb_ros,84.485,37.161,24.766,77.143,94.375,85.759,49.418,94.515,60.823


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,53.089,42.950,70.548,34.286,99.715,67.000,55.626,89.667,48.020
smote,82.054,42.743,33.339,72.143,95.454,83.798,48.788,91.997,62.399
bsmote,79.744,46.251,37.576,67.500,96.964,82.232,46.713,92.093,62.997
ros,82.481,44.252,34.624,72.857,95.833,84.345,46.851,91.363,63.367
adasyn,82.699,39.889,29.790,73.571,94.513,84.042,46.685,92.160,61.294
casb,81.688,45.157,35.767,71.071,96.273,83.672,48.345,91.647,63.423
casb_relaxed,81.092,43.526,33.893,70.357,95.997,83.177,49.499,91.995,62.309
casb_edited,83.951,41.524,29.876,75.357,95.083,85.220,49.279,92.397,62.738
casb_ros,81.846,45.944,36.739,71.429,96.394,83.911,48.205,91.642,63.895


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,86.685,ros,84.918,1.767,WIN
1,f1,casb_ros,50.379,bsmote,48.180,2.199,WIN
2,precision,casb_ros,37.875,bsmote,37.945,-0.070,LOSS
3,recall,casb_edited,78.571,ros,75.714,2.857,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,84.224,ros,83.576,0.649,WIN
1,f1,casb_ros,39.717,bsmote,45.874,-6.157,LOSS
2,precision,casb_ros,27.146,bsmote,33.971,-6.824,LOSS
3,recall,casb_edited,77.143,smote,74.286,2.857,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,78.852,adasyn,79.318,-0.466,LOSS
1,f1,casb_ros,56.518,ros,57.418,-0.900,LOSS
2,precision,casb_ros,57.169,ros,56.457,0.712,WIN
3,recall,casb_edited,65.714,adasyn,65.714,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,83.951,adasyn,82.699,1.252,WIN
1,f1,casb_ros,45.944,bsmote,46.251,-0.307,LOSS
2,precision,casb_ros,36.739,bsmote,37.576,-0.836,LOSS
3,recall,casb_edited,75.357,adasyn,73.571,1.786,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.377964,0.250000,1.000000,0.142857,1.000000,0.571429,0.660310,0.869951,0.313982,None
1,42,1,svm,smote,1.00,0.834892,0.526316,0.416667,0.714286,0.975862,0.845074,0.663080,0.962562,0.680604,None
2,42,1,svm,bsmote,0.50,0.748068,0.470588,0.400000,0.571429,0.979310,0.775369,0.679365,0.985714,0.609328,None
3,42,1,svm,ros,0.75,0.742781,0.380952,0.285714,0.571429,0.965517,0.768473,0.521849,0.972906,0.561867,None
4,42,1,svm,adasyn,0.50,0.834892,0.526316,0.416667,0.714286,0.975862,0.845074,0.638928,0.967980,0.680604,None
5,42,1,svm,casb,1.00,0.833415,0.500000,0.384615,0.714286,0.972414,0.843350,0.639469,0.972906,0.666708,None
6,42,1,svm,casb_relaxed,0.75,0.836366,0.555556,0.454545,0.714286,0.979310,0.846798,0.718421,0.972906,0.695961,None
7,42,1,svm,casb_edited,0.50,0.834892,0.526316,0.416667,0.714286,0.975862,0.845074,0.695342,0.977833,0.680604,None
8,42,1,svm,casb_ros,1.00,0.745429,0.421053,0.333333,0.571429,0.972414,0.771921,0.619873,0.980296,0.583241,None
9,42,1,knn,original,0.50,0.842235,0.714286,0.714286,0.714286,0.993103,0.853695,0.715136,0.982759,0.778260,None
